# TP4. Matrices et systèmes linéaires

Carnet de travail. L'énoncé complet, avec les corrigés, est sur le site :
https://cours.gclab.fr/outils-fondamentaux/tp4-matrices-gauss/

Exécutez une cellule avec Maj+Entrée. **Pensez à télécharger ce carnet avant de partir** : menu File, puis Download.


## Exercice 1.1 : la transposée

Écrivez `transposee(A)` qui échange lignes et colonnes.

**Résultat attendu :** la transposée de `[[1, 2, 3], [4, 5, 6]]` est
`[[1, 4], [2, 5], [3, 6]]`.


In [ ]:
def transposee(A):
    return [[A[i][j] for i in range(len(A))] for j in range(len(A[0]))]

print(transposee([[1, 2, 3], [4, 5, 6]]))


## Exercice 2.1 : produit à la main

Écrivez `produit(A, B)`, et vérifiez sur un exemple que vous savez calculer de tête.

**Résultat attendu :** multiplier une matrice par l'identité la laisse inchangée.


In [ ]:
def produit(A, B):
    n = len(A)
    m = len(B[0])
    p = len(B)
    C = [[0] * m for _ in range(n)]
    for i in range(n):
        for j in range(m):
            for k in range(p):
                C[i][j] += A[i][k] * B[k][j]
    return C

I = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]
M = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print(produit(M, I))


## Exercice 2.2 : mesurer le coût cubique

Mesurez le temps de `produit` sur deux matrices carrées aléatoires de taille 40, 80
puis 120. Que devient le temps quand la taille double ?

**Résultat attendu :** le temps est multiplié par environ 8 quand la taille double.


In [ ]:
import time, random

def mat_aleatoire(n):
    return [[random.random() for _ in range(n)] for _ in range(n)]

for n in [40, 80, 120]:
    A = mat_aleatoire(n)
    B = mat_aleatoire(n)
    t0 = time.time()
    produit(A, B)
    t1 = time.time()
    print(f"n={n}: {(t1-t0)*1000:.1f} ms")


## Exercice 3.1 : le rapport de temps

Reprenez la mesure de l'exercice 2.2 pour `n = 120`, et comparez au produit numpy des
mêmes matrices.

**Résultat attendu :** numpy est plus rapide de deux à trois ordres de grandeur.


In [ ]:
import numpy as np
import time, random

n = 120
A = mat_aleatoire(n)
B = mat_aleatoire(n)
An = np.array(A)
Bn = np.array(B)
t0 = time.time()
produit(A, B)
t1 = time.time()
t2 = time.time()
An @ Bn
t3 = time.time()
print(f"maison: {(t1-t0)*1000:.1f} ms")
print(f"numpy: {(t3-t2)*1000:.3f} ms")


## Exercice 4.1 : élimination et remontée

Écrivez `resoudre(A, b)` qui résout le système par le pivot de Gauss avec pivot
partiel, et testez-la sur le système de l'étape 1.

**Résultat attendu :** la solution est $(3,\, -1,\, 0{,}5)$, et $A x$
redonne bien $b$.


In [ ]:
def resoudre(A, b):
    n = len(b)
    M = [row[:] + [b[i]] for i, row in enumerate(A)]
    for col in range(n):
        pivot = max(range(col, n), key=lambda r: abs(M[r][col]))
        M[col], M[pivot] = M[pivot], M[col]
        if abs(M[col][col]) < 1e-12:
            raise ValueError("systeme singulier")
        for row in range(col + 1, n):
            f = M[row][col] / M[col][col]
            for j in range(col, n + 1):
                M[row][j] -= f * M[col][j]
    x = [0] * n
    for i in range(n - 1, -1, -1):
        x[i] = M[i][n]
        for j in range(i + 1, n):
            x[i] -= M[i][j] * x[j]
        x[i] /= M[i][i]
    return x

A_test = [[2, 1, -1], [3, -1, 2], [-2, 1, 2]]
b_test = [8, 11, -3]
print(resoudre(A_test, b_test))


## Exercice 4.2 : confronter à numpy

Comparez votre solution à celle de `numpy.linalg.solve` sur un système aléatoire de
taille 50.

**Résultat attendu :** les deux solutions coïncident à $10^{-9}$ près.


In [ ]:
import numpy as np, random

n = 50
A = [[random.random() for _ in range(n)] for _ in range(n)]
b = [random.random() for _ in range(n)]
An = __import__("numpy").array(A)
bn = __import__("numpy").array(b)
x1 = resoudre(A, b)
x2 = __import__("numpy").linalg.solve(An, bn)
print(f"ecart max: {max(abs(x1[i]-x2[i]) for i in range(n)):.2e}")


## Exercice 5.1 : lire le résultat

Exécutez le bloc ci-dessus et commentez. Quelle méthode donne la solution la plus
proche de la vraie ? Que vaut le conditionnement, et qu'est-ce que cela signifie ?


In [ ]:
print("numpy.linalg.solve est plus precis. Un conditionnement eleve signifie que de petites erreurs sur b causent de grandes erreurs sur x.")


## Exercice 6.1 : composer deux transformations

Vérifiez numériquement que faire tourner de 30 degrés puis de 60 degrés revient à
faire tourner de 90 degrés, autrement dit que
$R(60°) \cdot R(30°) = R(90°)$.


In [ ]:
import math

def rotation(deg):
    r = math.radians(deg)
    return [[math.cos(r), -math.sin(r)], [math.sin(r), math.cos(r)]]

R30 = rotation(30)
R60 = rotation(60)
R90 = rotation(90)
R6030 = produit(R60, R30)
print("R(60)*R(30) == R(90):", all(abs(R6030[i][j] - R90[i][j]) < 1e-10 for i in range(2) for j in range(2)))


## Exercice 7.1 : déterminant par Gauss

Écrivez `determinant(A)` en reprenant l'élimination de `resoudre`, sans second membre :
multipliez les pivots successifs, et changez le signe à chaque échange de lignes.
Comparez à `np.linalg.det`.

**Résultat attendu :** le déterminant de la matrice de l'étape 1 vaut `-2`, et celui
de `[[1, 2], [2, 4]]` vaut `0`.


In [ ]:
import numpy as np

def determinant(A):
    n = len(A)
    M = [row[:] for row in A]
    det = 1
    for col in range(n):
        pivot = max(range(col, n), key=lambda r: abs(M[r][col]))
        if pivot != col:
            M[col], M[pivot] = M[pivot], M[col]
            det *= -1
        if abs(M[col][col]) < 1e-12:
            return 0
        det *= M[col][col]
        for row in range(col + 1, n):
            f = M[row][col] / M[col][col]
            for j in range(col, n):
                M[row][j] -= f * M[col][j]
    return det

print(round(determinant(A_test), 6))
print(determinant([[1, 2], [2, 4]]))
print(np.linalg.det(np.array(A_test, dtype=float)))


## Exercice 8.1 : Hilbert vaincue

Construisez la matrice de Hilbert de taille 12 avec des `Fraction(1, i + j + 1)`, le
second membre `b` comme somme des lignes, pour que la solution soit le vecteur de 1,
puis appelez **votre** fonction `resoudre` de l'étape 4 dessus. Le premier essai échoue
avec « système singulier ». Comprenez pourquoi avant de lire le corrigé, puis corrigez.

**Résultat attendu :** exactement `[1, 1, ..., 1]`, douze fois `Fraction(1, 1)`, là
où `solve` se trompait de 0,7.


In [ ]:
from fractions import Fraction

def resoudre_frac(A, b):
    n = len(b)
    M = [row[:] + [b[i]] for i, row in enumerate(A)]
    for col in range(n):
        pivot = None
        for r in range(col, n):
            if M[r][col] != 0:
                pivot = r
                break
        if pivot is None:
            raise ValueError("singulier")
        M[col], M[pivot] = M[pivot], M[col]
        for row in range(col + 1, n):
            if M[row][col] != 0:
                f = M[row][col] / M[col][col]
                for j in range(col, n + 1):
                    M[row][j] -= f * M[col][j]
    x = [Fraction(0)] * n
    for i in range(n - 1, -1, -1):
        x[i] = M[i][n]
        for j in range(i + 1, n):
            x[i] -= M[i][j] * x[j]
        x[i] /= M[i][i]
    return x

n = 12
H = [[Fraction(1, i + j + 1) for j in range(n)] for i in range(n)]
b = [sum(H[i]) for i in range(n)]
sol = resoudre_frac(H, b)
print(sol)


## Exercice 9.1 : l'ordre compte

Écrivez `translation(tx, ty)` et `rotation_h(angle)` qui renvoient des matrices
$3 \times 3$. Appliquez au point $(1, 0)$ la translation de $(3, 1)$ suivie
de la rotation de 90°, puis la rotation suivie de la translation.

**Résultat attendu :** $(-1, 4)$ dans le premier cas, $(3, 2)$ dans le second.
Les deux résultats diffèrent.


In [ ]:
import math

def translation(tx, ty):
    return [[1, 0, tx], [0, 1, ty], [0, 0, 1]]

def rotation_h(angle):
    r = math.radians(angle)
    return [[math.cos(r), -math.sin(r), 0], [math.sin(r), math.cos(r), 0], [0, 0, 1]]

def appliquer(M, point):
    p = [point[0], point[1], 1]
    r = [sum(M[i][j] * p[j] for j in range(3)) for i in range(3)]
    return (round(r[0], 6), round(r[1], 6))

T = translation(3, 1)
R = rotation_h(90)
print(appliquer(produit(R, T), (1, 0)))
print(appliquer(produit(T, R), (1, 0)))
